# Aplicación de los modelos seleccionados hasta 2028
Origen fijo 2026Q1. Las bandas tienen niveles nominales del 80 % y 95 %; consultar su cobertura reciente antes de interpretarlas. Cada horizonte hereda la especificación de la fase 1.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'sources.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'sources.json').is_file(), 'Abrir el cuaderno desde el repositorio o notebooks/'
REPORTS = ROOT / 'data/reports/unified'
EDA = ROOT / 'data/reports/eda'
pd.set_option('display.max_columns', 20)

In [ ]:
forecast = pd.read_csv(ROOT / 'data/forecasts/prevision_provincial_2026T2_2028T4.csv', dtype={'province_code': str}).rename(columns={'provincia': 'province', 'trimestre': 'forecast_quarter', 'prediccion': 'prediction', 'limite_inferior_80': 'lower_80', 'limite_superior_80': 'upper_80', 'limite_inferior_95': 'lower_95', 'limite_superior_95': 'upper_95'})
selection = json.loads((REPORTS / 'selection.json').read_text(encoding='utf-8'))
if 'selection_id' in forecast:
    assert set(forecast.selection_id) == {selection['selection_id']}
display(forecast.head())
display(forecast[forecast.forecast_quarter.eq('2028Q4')])

In [ ]:
province_code = '28'  # Cambiar por otro código INE de dos caracteres
part = forecast.loc[forecast.province_code.str.zfill(2).eq(province_code)].sort_values('horizon')
ax = part.plot(x='horizon', y='prediction', marker='o', figsize=(10, 5), legend=False)
for confidence, alpha in [(95, .15), (80, .25)]:
    low, high = f'lower_{confidence}', f'upper_{confidence}'
    if low in part and high in part:
        ax.fill_between(part.horizon, part[low], part[high], alpha=alpha, label=f'Banda nominal {confidence} %')
ax.set(title=str(part.province.iloc[0]), xlabel='Trimestres desde 2026Q1', ylabel='Valor tasado previsto (€/m²)')
ax.legend()
plt.show()

## Coherencia de la trayectoria
Los modelos independientes por horizonte pueden generar cambios entre puntos consecutivos. No se suavizan después de evaluar. Esta previsión no incorpora información publicada después del origen fijo ni describe un intervalo nacional.

In [ ]:
path = REPORTS / 'forecast_trajectory_diagnostics.csv'
if path.is_file():
    display(pd.read_csv(path))